# Quota sampling

Where a guidance forecast sets a group's count for a month, a domain draws exactly that many
rows, the model's probabilities choosing which ([ADR 0002](../adr/0002-quota-sampling.md)). This
notebook sets that beside the alternative it replaces, drawing every row on its own at
probabilities that sum to the same count, using the samplers the engine itself calls
(`hcsim.model.ml.sampling`).

Two hundred rows, with probabilities that vary as much as a model's might, sum to a quota of 20.

In [ ]:
import numpy as np
import pandas as pd

from hcsim.model.ml.sampling import bernoulli, quota_select

rng = np.random.default_rng(0)
rows, quota, draws = 200, 20, 2_000
weights = rng.gamma(1.0, 1.0, rows)  # a coefficient of variation of 1
p = np.minimum(weights * quota / weights.sum(), 1.0)
groups = np.zeros(rows, dtype=np.int64)
print(f"the rows' probabilities sum to {p.sum():.2f}")

Each draw, quota sampling realizes exactly the quota, while independent draws only expect it.

In [ ]:
counts = {"quota sampling": [], "independent draws": []}
hits = {name: np.zeros(rows) for name in counts}
for _ in range(draws):
    chosen = {
        "quota sampling": quota_select(p, groups, np.array([quota]), rng).mask,
        "independent draws": bernoulli(p, rng),
    }
    for name, mask in chosen.items():
        counts[name].append(int(mask.sum()))
        hits[name] += mask

summary = pd.DataFrame(
    {
        name: {"mean": np.mean(c), "sd": np.std(c, ddof=1), "min": min(c), "max": max(c)}
        for name, c in counts.items()
    }
).T
summary

In [ ]:
assert set(counts["quota sampling"]) == {quota}
assert abs(np.mean(counts["independent draws"]) - quota) < 0.5

The count is fixed, but *who* is drawn still follows the model: under both samplers, a row's
share of the draws rises with its probability. Quota sampling's shares aren't exactly the
probabilities, since it draws without replacement within the month, but they rank the rows the
same way.

In [ ]:
shares = pd.DataFrame({"probability": p} | {name: h / draws for name, h in hits.items()})
deciles = pd.qcut(shares["probability"].rank(method="first"), 10, labels=range(1, 11))
by_decile = shares.groupby(deciles, observed=True).mean()
by_decile

In [ ]:
rank = shares.corr(method="spearman")["probability"]
assert rank["quota sampling"] > 0.8
assert rank["independent draws"] > 0.8
rank

So a guided run's totals match the forecast in every group and month by construction, and only
which employees leave, fill, or move stays random. The `why_quota_sampling` study in the
catalog measures the same thing on a whole run, against history.